In [1]:
x=1

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
from sklearn.model_selection import KFold

/lustre/groups/ml01/workspace/anastasia.litinetskaya/mambaforge/envs/sc/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/lustre/groups/ml01/workspace/anastasia.litinetskaya/mambaforge/envs/sc/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/lustre/groups/ml01/workspace/anastasia.litinetskaya/mambaforge/envs/sc/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/lustre/groups/ml01/workspace/anastasia.litinetskaya/mambaforge/envs/sc/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx ins

In [3]:
df = pd.read_csv('/lustre/groups/ml01/workspace/mayar.ali/tissue/data/jackson/Data_publication/ZurichTMA/SC_dat.csv')

In [4]:
df

,core,CellId,id,channel,mc_counts
0,ZTMA208_slide_10_By10x4,1,ZTMA208_slide_10_By10x4_1,1021522Tm169Di EGFR,0.450014
1,ZTMA208_slide_10_By10x4,10,ZTMA208_slide_10_By10x4_10,1021522Tm169Di EGFR,0.090965
2,ZTMA208_slide_10_By10x4,100,ZTMA208_slide_10_By10x4_100,1021522Tm169Di EGFR,0.205657
3,ZTMA208_slide_10_By10x4,1000,ZTMA208_slide_10_By10x4_1000,1021522Tm169Di EGFR,0.665907
4,ZTMA208_slide_10_By10x4,1001,ZTMA208_slide_10_By10x4_1001,1021522Tm169Di EGFR,0.571331
...,...,...,...,...,...
29452631,ZTMA208_slide_non-breast_Dy3x0,95,ZTMA208_slide_non-breast_Dy3x0_95,Number_Neighbors,3.000000
29452632,ZTMA208_slide_non-breast_Dy3x0,96,ZTMA208_slide_non-breast_Dy3x0_96,Number_Neighbors,3.000000
29452633,ZTMA208_slide_non-breast_Dy3x0,97,ZTMA208_slide_non-breast_Dy3x0_97,Number_Neighbors,2.000000
29452634,ZTMA208_slide_non-breast_Dy3x0,98,ZTMA208_slide_non-breast_Dy3x0_98,Number_Neighbors,2.000000


In [5]:
len(set(df['id']))

433127

In [6]:
# add a column 'core' which is the same as 'id' but without the last number, i.e. split by '_' and join all but the last element
df['core'] = df['id'].apply(lambda x: '_'.join(x.split('_')[:-1]))

In [7]:
len(np.unique(df['core']))

365

In [8]:
metadata = pd.read_csv('/lustre/groups/ml01/workspace/mayar.ali/tissue/data/jackson/Data_publication/ZurichTMA/Zuri_PatientMetadata.csv')

In [9]:
metadata

,PID,core,location,grade,tumor_type,tumor_size,gender,menopausal,PTNM_M,age,...,FileName_FullStack,Height_FullStack,Width_FullStack,area,sum_area_cells,Subtype,HER2,ER,PR,clinical_type
0,6,ZTMA208_slide_6_Ay4x3,CENTER,1,INVASIVE_DUCTAL_LOBULAR,30.0,FEMALE,POST,M1,47,...,ZTMA208_slide_28.23kx15.4ky_7000x7000_4_201711...,542,346,187532,46669,PR+ER+,+,+,+,HR+HER2+
1,6,ZTMA208_slide_6_Ay4x4,CENTER,1,INVASIVE_DUCTAL_LOBULAR,30.0,FEMALE,POST,M1,47,...,ZTMA208_slide_28.23kx15.4ky_7000x7000_4_201711...,488,454,221552,38958,PR+ER+,+,+,+,HR+HER2+
2,6,ZTMA208_slide_6_Ay4x6,PERIPHERY,1,INVASIVE_DUCTAL_LOBULAR,30.0,FEMALE,POST,M1,47,...,ZTMA208_slide_28.23kx15.4ky_7000x7000_4_201711...,454,614,278756,60250,PR+ER+,+,+,+,HR+HER2+
3,6,ZTMA208_slide_6_Ay4x7,[],1,[],NaN,FEMALE,POST,M1,47,...,ZTMA208_slide_28.23kx15.4ky_7000x7000_4_201711...,481,444,213564,3064,PR+ER+,+,+,+,HR+HER2+
4,12,ZTMA208_slide_12_Ay13x7,CENTER,2,INVASIVE_DUCTAL,25.0,FEMALE,POST,M1,42,...,ZTMA208_slide_28.23kx22.4ky_7000x7000_5_201711...,497,519,257943,223730,PR+ER+,+,+,+,HR+HER2+
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
354,16,ZTMA208_slide_16_Ay16x5,CENTER,2,INVASIVE_DUCTAL,55.0,FEMALE,POST,M1,59,...,ZTMA208_slide_28.23kx22.4ky_7000x7000_5_201711...,503,475,238925,98597,PR+ER+,NaN,NaN,NaN,NaN
355,16,ZTMA208_slide_16_Ay16x6,PERIPHERY,2,INVASIVE_DUCTAL,55.0,FEMALE,POST,M1,59,...,ZTMA208_slide_28.23kx22.4ky_7000x7000_5_201711...,450,541,243450,102638,PR+ER+,NaN,NaN,NaN,NaN
356,16,ZTMA208_slide_16_Ay16x7,PERIPHERY,2,INVASIVE_DUCTAL,55.0,FEMALE,POST,M1,59,...,ZTMA208_slide_28.23kx22.4ky_7000x7000_5_201711...,464,519,240816,123359,PR+ER+,NaN,NaN,NaN,NaN
357,16,ZTMA208_slide_16_By1x1,METASTASIS,METASTASIS,[],NaN,FEMALE,POST,M1,59,...,ZTMA208_slide_20.73kx15.16ky_7000x7000_6_20171...,488,540,263520,241444,PR+ER+,NaN,NaN,NaN,NaN


In [10]:
set(df['core']) - set(metadata['core'])

{'ZTMA208_slide_control_Cy15x6',
 'ZTMA208_slide_control_Cy15x7',
 'ZTMA208_slide_control_Cy15x8',
 'ZTMA208_slide_non-breast_Dy1x0',
 'ZTMA208_slide_non-breast_Dy2x0',
 'ZTMA208_slide_non-breast_Dy3x0'}

In [11]:
channels = list(set(df['channel']))
len(channels)

68

In [12]:
exclude_channels = ["ImageId" ,"CellId" ,"In115 115InIn115Di","Xe134 134XeXe134Di","Hg202 202HgHg202Di","Pb204 204PbPb204Di","Pb206 206PbPb206Di","ArAr80 80ArArArAr80Di", "10311239Ru96Di Rutheni","10311240Ru98Di Rutheni","10311241Ru99Di Rutheni", "10311242Ru100Di Rutheni","10311243Ru101Di Rutheni", "10311244Ru102Di Rutheni","10311245Ru104Di Rutheni","Xe126 126XeXe126Di","I127 127II127Di","Xe131 131XeXe131Di","Pb207 207PbPb207Di","Pb208 208PbPb208Di","EulerNumber","MajorAxisLength","MinorAxisLength", "Orientation","10331253Ir191Di Iridium","Perimeter","Solidity",
'Area', 'Eccentricity', 'Extent', 'Number_Neighbors', 'Percent_Touching',]
len(exclude_channels)

32

In [13]:
# check which cnannels are in the exclude list but not in the data
set(exclude_channels) - set(channels)

{'CellId', 'ImageId'}

In [14]:
len(set(channels) - set(exclude_channels))

38

In [15]:
# reformat df to have one row per 'id', with columns for each channel
data_df = df.pivot(index='id', columns='channel', values='mc_counts').reset_index()
# exclude columns from exclude_channels if present 
data_df = data_df.drop(columns=exclude_channels, errors='ignore')
# set index to id
data_df = data_df.set_index('id')
data_df

channel,1021522Tm169Di EGFR,1031747Er167Di ECadhe,10331254Ir193Di Iridium,112475Gd156Di Estroge,117792Dy163Di GATA3,1261726In113Di Histone,1441101Er168Di Ki67,174864Nd148Di SMA,1921755Sm149Di Vimenti,1971527Ho165Di bCaten,...,77877Nd146Di CD68,8001752Sm152Di CD3epsi,92964Er166Di Carboni,971099Nd144Di Cytoker,98922Yb174Di Cytoker,Nd145Di Twist,phospho Erk12,phospho Histone,phospho S6,phospho mTOR
id,,,,,,,,,,,,,,,,,,,,,
ZTMA208_slide_10_By10x4_1,0.450014,1.245225,8.031889,0.161779,0.404193,4.405373,0.209200,1.588501,1.167121,0.077193,...,0.079376,0.124376,0.249224,0.221780,0.315186,0.314232,0.479315,0.200821,0.297848,0.701305
ZTMA208_slide_10_By10x4_10,0.090965,0.391418,6.530071,0.140406,0.040895,1.897472,0.000000,0.291961,1.321693,0.023715,...,0.012012,0.108462,0.067022,0.018082,0.000000,0.203987,0.042693,0.022841,0.000000,0.166950
ZTMA208_slide_10_By10x4_100,0.205657,0.464244,5.788927,0.124095,0.013957,0.863905,0.012968,0.170728,2.055171,0.146121,...,0.156818,0.014493,0.216740,0.011685,0.000000,0.303555,0.087335,0.041179,0.329447,0.376549
ZTMA208_slide_10_By10x4_1000,0.665907,9.560929,8.492696,0.406356,1.152035,11.426278,0.123461,0.842307,0.713615,0.478272,...,0.111560,0.163419,0.989136,1.935666,17.678322,0.686220,0.305358,0.121409,0.478091,1.756742
ZTMA208_slide_10_By10x4_1001,0.571331,1.747541,13.949946,0.633572,0.295532,10.399852,0.039642,7.924132,27.976877,0.232022,...,0.471818,0.215325,0.773444,0.167445,0.000000,0.678661,0.412644,0.442049,0.659149,1.233168
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZTMA208_slide_non-breast_Dy3x0_95,0.135308,0.159011,3.396363,0.025914,0.061319,3.251136,0.006011,0.577265,0.024152,0.021180,...,0.097910,0.050297,0.096396,0.326195,0.021482,0.064966,0.277286,0.053407,0.000000,0.235934
ZTMA208_slide_non-breast_Dy3x0_96,0.118098,0.070268,4.453231,0.012528,0.003621,1.668683,0.019656,1.495501,0.033777,0.016840,...,0.092282,0.014904,0.045672,0.019450,0.007082,0.056599,0.092229,0.068336,0.026292,0.065263
ZTMA208_slide_non-breast_Dy3x0_97,0.081961,0.029598,5.836277,0.015977,0.025735,2.105749,0.029413,0.660908,0.036041,0.044884,...,0.023621,0.043911,0.047462,0.036157,0.007291,0.049714,0.252056,0.016323,0.000000,0.105000


In [16]:
data_df.columns

Index(['1021522Tm169Di EGFR', '1031747Er167Di ECadhe',
       '10331254Ir193Di Iridium', '112475Gd156Di Estroge',
       '117792Dy163Di GATA3', '1261726In113Di Histone', '1441101Er168Di Ki67',
       '174864Nd148Di SMA', '1921755Sm149Di Vimenti', '1971527Ho165Di bCaten',
       '198883Yb176Di cleaved', '201487Eu151Di cerbB', '207736Tb159Di p53',
       '234832Lu175Di panCyto', '2971330Dy161Di EpCAM',
       '3111576Nd143Di Cytoker', '312878Gd158Di Progest', '322787Nd150Di cMyc',
       '3281668Nd142Di Fibrone', '346876Sm147Di Keratin',
       '3521227Gd155Di Slug', '361077Dy164Di CD20', '378871Yb172Di vWF',
       '473968La139Di Histone', '483739Yb171Di Sox9', '651779Pr141Di Cytoker',
       '6967Gd160Di CD44', '71790Dy162Di CD45', '77877Nd146Di CD68',
       '8001752Sm152Di CD3epsi', '92964Er166Di Carboni',
       '971099Nd144Di Cytoker', '98922Yb174Di Cytoker', 'Nd145Di Twist',
       'phospho Erk12', 'phospho Histone', 'phospho S6', 'phospho mTOR'],
      dtype='object', name='chann

In [17]:
# create another df with one row per 'id' whith the values form metadata; get 'core' from the index of data_df as above and join on 'core' in metadata
# don't keep any of the columns from data_df
obs = data_df.reset_index()
obs['core'] = obs['id'].apply(lambda x: '_'.join(x.split('_')[:-1]))
obs = obs.merge(metadata, on='core')
obs = obs.drop(columns=data_df.columns)
obs

,id,core,PID,location,grade,tumor_type,tumor_size,gender,menopausal,PTNM_M,...,FileName_FullStack,Height_FullStack,Width_FullStack,area,sum_area_cells,Subtype,HER2,ER,PR,clinical_type
0,ZTMA208_slide_10_By10x4_1,ZTMA208_slide_10_By10x4,10,CENTER,2,INVASIVE_LOBULAR,40.0,FEMALE,POST,M1,...,ZTMA208_slide_20.73kx22.16ky_7000x7000_7_20171...,445,513,228285,94975,PR+ER+,NaN,NaN,NaN,NaN
1,ZTMA208_slide_10_By10x4_10,ZTMA208_slide_10_By10x4,10,CENTER,2,INVASIVE_LOBULAR,40.0,FEMALE,POST,M1,...,ZTMA208_slide_20.73kx22.16ky_7000x7000_7_20171...,445,513,228285,94975,PR+ER+,NaN,NaN,NaN,NaN
2,ZTMA208_slide_10_By10x4_100,ZTMA208_slide_10_By10x4,10,CENTER,2,INVASIVE_LOBULAR,40.0,FEMALE,POST,M1,...,ZTMA208_slide_20.73kx22.16ky_7000x7000_7_20171...,445,513,228285,94975,PR+ER+,NaN,NaN,NaN,NaN
3,ZTMA208_slide_10_By10x4_1000,ZTMA208_slide_10_By10x4,10,CENTER,2,INVASIVE_LOBULAR,40.0,FEMALE,POST,M1,...,ZTMA208_slide_20.73kx22.16ky_7000x7000_7_20171...,445,513,228285,94975,PR+ER+,NaN,NaN,NaN,NaN
4,ZTMA208_slide_10_By10x4_1001,ZTMA208_slide_10_By10x4,10,CENTER,2,INVASIVE_LOBULAR,40.0,FEMALE,POST,M1,...,ZTMA208_slide_20.73kx22.16ky_7000x7000_7_20171...,445,513,228285,94975,PR+ER+,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427261,ZTMA208_slide_9_By4x5_995,ZTMA208_slide_9_By4x5,9,METASTASIS,METASTASIS,[],NaN,FEMALE,POST,M1,...,ZTMA208_slide_20.73kx15.16ky_7000x7000_6_20171...,480,522,250560,201565,PR+ER+,-,+,+,HR+HER2-
427262,ZTMA208_slide_9_By4x5_996,ZTMA208_slide_9_By4x5,9,METASTASIS,METASTASIS,[],NaN,FEMALE,POST,M1,...,ZTMA208_slide_20.73kx15.16ky_7000x7000_6_20171...,480,522,250560,201565,PR+ER+,-,+,+,HR+HER2-
427263,ZTMA208_slide_9_By4x5_997,ZTMA208_slide_9_By4x5,9,METASTASIS,METASTASIS,[],NaN,FEMALE,POST,M1,...,ZTMA208_slide_20.73kx15.16ky_7000x7000_6_20171...,480,522,250560,201565,PR+ER+,-,+,+,HR+HER2-
427264,ZTMA208_slide_9_By4x5_998,ZTMA208_slide_9_By4x5,9,METASTASIS,METASTASIS,[],NaN,FEMALE,POST,M1,...,ZTMA208_slide_20.73kx15.16ky_7000x7000_6_20171...,480,522,250560,201565,PR+ER+,-,+,+,HR+HER2-


In [18]:
# drop samples with core values from set(df['core']) - set(metadata['core'])
data_df = data_df.loc[data_df.index.isin(obs['id'])]
obs = obs.loc[obs['id'].isin(data_df.index)]
data_df

channel,1021522Tm169Di EGFR,1031747Er167Di ECadhe,10331254Ir193Di Iridium,112475Gd156Di Estroge,117792Dy163Di GATA3,1261726In113Di Histone,1441101Er168Di Ki67,174864Nd148Di SMA,1921755Sm149Di Vimenti,1971527Ho165Di bCaten,...,77877Nd146Di CD68,8001752Sm152Di CD3epsi,92964Er166Di Carboni,971099Nd144Di Cytoker,98922Yb174Di Cytoker,Nd145Di Twist,phospho Erk12,phospho Histone,phospho S6,phospho mTOR
id,,,,,,,,,,,,,,,,,,,,,
ZTMA208_slide_10_By10x4_1,0.450014,1.245225,8.031889,0.161779,0.404193,4.405373,0.209200,1.588501,1.167121,0.077193,...,0.079376,0.124376,0.249224,0.221780,0.315186,0.314232,0.479315,0.200821,0.297848,0.701305
ZTMA208_slide_10_By10x4_10,0.090965,0.391418,6.530071,0.140406,0.040895,1.897472,0.000000,0.291961,1.321693,0.023715,...,0.012012,0.108462,0.067022,0.018082,0.000000,0.203987,0.042693,0.022841,0.000000,0.166950
ZTMA208_slide_10_By10x4_100,0.205657,0.464244,5.788927,0.124095,0.013957,0.863905,0.012968,0.170728,2.055171,0.146121,...,0.156818,0.014493,0.216740,0.011685,0.000000,0.303555,0.087335,0.041179,0.329447,0.376549
ZTMA208_slide_10_By10x4_1000,0.665907,9.560929,8.492696,0.406356,1.152035,11.426278,0.123461,0.842307,0.713615,0.478272,...,0.111560,0.163419,0.989136,1.935666,17.678322,0.686220,0.305358,0.121409,0.478091,1.756742
ZTMA208_slide_10_By10x4_1001,0.571331,1.747541,13.949946,0.633572,0.295532,10.399852,0.039642,7.924132,27.976877,0.232022,...,0.471818,0.215325,0.773444,0.167445,0.000000,0.678661,0.412644,0.442049,0.659149,1.233168
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZTMA208_slide_9_By4x5_995,0.678043,1.366533,6.998387,2.534251,1.774004,11.645180,0.090557,0.172837,0.914335,0.083079,...,0.192483,0.108202,1.132215,10.545009,3.276609,0.802634,0.334068,1.635917,0.992162,14.618638
ZTMA208_slide_9_By4x5_996,0.929184,1.237893,4.035522,3.049171,1.043037,8.613761,0.076334,0.344354,4.651445,0.207099,...,0.135342,0.193577,1.119748,7.214287,0.782363,0.676726,0.406478,0.997655,1.072101,13.254979
ZTMA208_slide_9_By4x5_997,0.590335,1.081519,4.259664,0.458342,0.438101,5.336907,0.088053,0.115044,2.688884,0.202616,...,0.687869,0.129045,0.622456,5.743281,0.883737,0.259990,0.548990,0.549772,0.523019,7.706533


In [19]:
obs.index = obs['id']

In [20]:
# keep only the columns that are present in both data_df and obs
common_obs = set(obs.index) & set(data_df.index)
len(common_obs)

427266

In [21]:
del df

In [22]:
len(data_df.index)

427266

In [23]:
len(obs.index)

427266

In [24]:
adata = ad.AnnData(X=data_df.loc[list(common_obs)], obs=obs.loc[list(common_obs)])
adata

AnnData object with n_obs × n_vars = 427266 × 38
    obs: 'id', 'core', 'PID', 'location', 'grade', 'tumor_type', 'tumor_size', 'gender', 'menopausal', 'PTNM_M', 'age', 'Patientstatus', 'treatment', 'PTNM_T', 'DiseaseStage', 'PTNM_N', 'AllSamplesSVSp4.Array', 'TMALocation', 'TMAxlocation', 'yLocation', 'DonorBlocklabel', 'diseasestatus', 'TMABlocklabel', 'UBTMAlocation', '%tumorcells', '%normalepithelialcells', '%stroma', '%inflammatorycells', 'SupplierPatientID', 'Yearofsamplecollection', 'PrimarySite', 'histology', 'PTNM_Radicality', 'Lymphaticinvasion', 'Venousinvasion', 'ERStatus', 'PRStatus', 'HER2Status', 'Pre-surgeryTx', 'Post-surgeryTx', 'Tag', 'Ptdiagnosis', 'DFSmonth', 'OSmonth', 'Comment', 'ER+DuctalCa', 'TripleNegDuctal', 'hormonesensitive', 'hormonerefractory', 'hormoneresistantaftersenstive', 'Fulvestran', 'microinvasion', 'I_plus_neg', 'SN', 'MIC', 'Count_Cells', 'FileName_FullStack', 'Height_FullStack', 'Width_FullStack', 'area', 'sum_area_cells', 'Subtype', 'HER2', 'ER

In [ ]:
adata.write('data/zurich.h5ad')

In [37]:
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [44]:
adata.obs.columns

Index(['id', 'core', 'PID', 'location', 'grade', 'tumor_type', 'tumor_size',
       'gender', 'menopausal', 'PTNM_M', 'age', 'Patientstatus', 'treatment',
       'PTNM_T', 'DiseaseStage', 'PTNM_N', 'AllSamplesSVSp4.Array',
       'TMALocation', 'TMAxlocation', 'yLocation', 'DonorBlocklabel',
       'diseasestatus', 'TMABlocklabel', 'UBTMAlocation', '%tumorcells',
       '%normalepithelialcells', '%stroma', '%inflammatorycells',
       'SupplierPatientID', 'Yearofsamplecollection', 'PrimarySite',
       'histology', 'PTNM_Radicality', 'Lymphaticinvasion', 'Venousinvasion',
       'ERStatus', 'PRStatus', 'HER2Status', 'Pre-surgeryTx', 'Post-surgeryTx',
       'Tag', 'Ptdiagnosis', 'DFSmonth', 'OSmonth', 'Comment', 'ER+DuctalCa',
       'TripleNegDuctal', 'hormonesensitive', 'hormonerefractory',
       'hormoneresistantaftersenstive', 'Fulvestran', 'microinvasion',
       'I_plus_neg', 'SN', 'MIC', 'Count_Cells', 'FileName_FullStack',
       'Height_FullStack', 'Width_FullStack', 'area', 

In [47]:
sc.pl.umap(adata, color=['location', 'grade', 'tumor_type', 'tumor_size',
       'gender', 'menopausal', 'PTNM_M', 'age', 'Patientstatus', 'treatment',
       'PTNM_T', 'DiseaseStage', 'PTNM_N', 'AllSamplesSVSp4.Array',
       'TMALocation', 'TMAxlocation', 'yLocation', 'DonorBlocklabel',
       'diseasestatus', 'TMABlocklabel', 'UBTMAlocation', '%tumorcells',
       '%normalepithelialcells', '%stroma', '%inflammatorycells',
       'SupplierPatientID', 'Yearofsamplecollection', 'PrimarySite',
       'histology', 'PTNM_Radicality', 'Lymphaticinvasion', 'Venousinvasion',
       'ERStatus', 'PRStatus', 'HER2Status', 'Pre-surgeryTx', 'Post-surgeryTx',
       'Tag', 'Ptdiagnosis', 'DFSmonth', 'OSmonth', 'Comment', 'ER+DuctalCa',
       'TripleNegDuctal', 'hormonesensitive', 'hormonerefractory',
       'hormoneresistantaftersenstive', 'Fulvestran', 'microinvasion',
       'I_plus_neg', 'SN', 'MIC', 'Count_Cells', 'FileName_FullStack',
       'Height_FullStack', 'Width_FullStack', 'area', 'sum_area_cells',
       'Subtype', 'HER2', 'ER', 'PR', 'clinical_type'], ncols=1, frameon=False)

# split

## subtypes

In [33]:
# check how many cells are there for each subtype
adata.obs['Subtype'].value_counts()

Subtype
PR+ER+    331932
PR-ER+     44383
PR-ER-     36678
PR+ER-     14273
Name: count, dtype: int64

In [34]:
# check how many patients are there for each subtype
adata.obs[['core', 'Subtype']].drop_duplicates()['Subtype'].value_counts()

Subtype
PR+ER+    275
PR-ER+     49
PR-ER-     25
PR+ER-     10
Name: count, dtype: int64

In [50]:
# subsample adata to have fewer PR+ER+ Subtype patients
# sample 10 patients with PR+ER+ subtype
patients_to_keep = list(adata.obs[adata.obs['Subtype'] == 'PR+ER+']['core'].sample(50).values)
# add all patients with other subtypes
patients_to_keep.extend(np.unique(adata.obs[adata.obs['Subtype'] != 'PR+ER+']['core'].values))
len(patients_to_keep)

134

In [51]:
# subset adata to keep only patients in patients_to_keep
adata_subtype = adata[adata.obs['core'].isin(patients_to_keep)].copy()
adata_subtype

AnnData object with n_obs × n_vars = 182041 × 38
    obs: 'id', 'core', 'PID', 'location', 'grade', 'tumor_type', 'tumor_size', 'gender', 'menopausal', 'PTNM_M', 'age', 'Patientstatus', 'treatment', 'PTNM_T', 'DiseaseStage', 'PTNM_N', 'AllSamplesSVSp4.Array', 'TMALocation', 'TMAxlocation', 'yLocation', 'DonorBlocklabel', 'diseasestatus', 'TMABlocklabel', 'UBTMAlocation', '%tumorcells', '%normalepithelialcells', '%stroma', '%inflammatorycells', 'SupplierPatientID', 'Yearofsamplecollection', 'PrimarySite', 'histology', 'PTNM_Radicality', 'Lymphaticinvasion', 'Venousinvasion', 'ERStatus', 'PRStatus', 'HER2Status', 'Pre-surgeryTx', 'Post-surgeryTx', 'Tag', 'Ptdiagnosis', 'DFSmonth', 'OSmonth', 'Comment', 'ER+DuctalCa', 'TripleNegDuctal', 'hormonesensitive', 'hormonerefractory', 'hormoneresistantaftersenstive', 'Fulvestran', 'microinvasion', 'I_plus_neg', 'SN', 'MIC', 'Count_Cells', 'FileName_FullStack', 'Height_FullStack', 'Width_FullStack', 'area', 'sum_area_cells', 'Subtype', 'HER2', 'ER

In [52]:
adata_subtype.obs[['core', 'Subtype']].drop_duplicates()['Subtype'].value_counts()

Subtype
PR-ER+    49
PR+ER+    46
PR-ER-    25
PR+ER-    10
Name: count, dtype: int64

In [53]:
# check how many patients are there for each grade
adata.obs[['core', 'grade']].drop_duplicates()['grade'].value_counts()

grade
3             130
2             122
1              95
METASTASIS     12
Name: count, dtype: int64

In [54]:
samples = np.unique(adata_subtype.obs['core'])
len(samples)

130

In [55]:
kf = KFold(n_splits=5, random_state=1, shuffle=True)

In [57]:
train_test_sizes = []

for i, (train_index, test_index) in enumerate(kf.split(samples)):
    train_samples = samples[train_index]
    test_samples = samples[test_index]
    adata_subtype.obs.loc[adata_subtype.obs['core'].isin(train_samples), f'split{i}'] = 'train'
    adata_subtype.obs.loc[adata_subtype.obs['core'].isin(test_samples), f'split{i}'] = 'val'
    adata_train = adata_subtype[adata_subtype.obs[f'split{i}'] == 'train'].copy()
    adata_test = adata_subtype[adata_subtype.obs[f'split{i}'] == 'val'].copy()
    train_conditions = set(np.unique(adata_train.obs['Subtype']))
    test_conditions = set(np.unique(adata_test.obs['Subtype']))
    print(train_conditions)
    print(test_conditions)
    assert test_conditions.issubset(train_conditions)
    train_test_sizes.append((len(adata_train), len(adata_test)))

{'PR-ER+', 'PR+ER+', 'PR+ER-', 'PR-ER-'}
{'PR-ER+', 'PR+ER+', 'PR+ER-', 'PR-ER-'}
{'PR-ER+', 'PR+ER+', 'PR+ER-', 'PR-ER-'}
{'PR-ER+', 'PR+ER+', 'PR+ER-', 'PR-ER-'}
{'PR-ER+', 'PR+ER+', 'PR+ER-', 'PR-ER-'}
{'PR-ER+', 'PR+ER+', 'PR+ER-', 'PR-ER-'}
{'PR-ER+', 'PR+ER+', 'PR+ER-', 'PR-ER-'}
{'PR-ER+', 'PR+ER+', 'PR+ER-', 'PR-ER-'}
{'PR-ER+', 'PR+ER+', 'PR+ER-', 'PR-ER-'}
{'PR-ER+', 'PR+ER+', 'PR+ER-', 'PR-ER-'}


In [58]:
adata_subtype.write('data/zurich_subtype.h5ad')

## grade

In [25]:
samples = np.unique(adata.obs['core'])
len(samples)

359

In [26]:
kf = KFold(n_splits=5, random_state=1, shuffle=True)

In [27]:
train_test_sizes = []

for i, (train_index, test_index) in enumerate(kf.split(samples)):
    train_samples = samples[train_index]
    test_samples = samples[test_index]
    adata.obs.loc[adata.obs['core'].isin(train_samples), f'split{i}'] = 'train'
    adata.obs.loc[adata.obs['core'].isin(test_samples), f'split{i}'] = 'val'
    adata_train = adata[adata.obs[f'split{i}'] == 'train'].copy()
    adata_test = adata[adata.obs[f'split{i}'] == 'val'].copy()
    train_conditions = set(np.unique(adata_train.obs['grade']))
    test_conditions = set(np.unique(adata_test.obs['grade']))
    print(train_conditions)
    print(test_conditions)
    assert test_conditions.issubset(train_conditions)
    train_test_sizes.append((len(adata_train), len(adata_test)))

{'2', '3', '1', 'METASTASIS'}
{'2', '3', '1', 'METASTASIS'}
{'2', '3', '1', 'METASTASIS'}
{'2', '3', '1', 'METASTASIS'}
{'2', '3', '1', 'METASTASIS'}
{'2', '3', '1', 'METASTASIS'}
{'2', '3', '1', 'METASTASIS'}
{'2', '3', '1', 'METASTASIS'}
{'2', '3', '1', 'METASTASIS'}
{'2', '3', '1', 'METASTASIS'}


In [28]:
adata.write('data/zurich_grade.h5ad')

## grade no met

In [3]:
adata = sc.read('data/zurich_grade.h5ad')
adata

AnnData object with n_obs × n_vars = 427266 × 38
    obs: 'id', 'core', 'PID', 'location', 'grade', 'tumor_type', 'tumor_size', 'gender', 'menopausal', 'PTNM_M', 'age', 'Patientstatus', 'treatment', 'PTNM_T', 'DiseaseStage', 'PTNM_N', 'AllSamplesSVSp4.Array', 'TMALocation', 'TMAxlocation', 'yLocation', 'DonorBlocklabel', 'diseasestatus', 'TMABlocklabel', 'UBTMAlocation', '%tumorcells', '%normalepithelialcells', '%stroma', '%inflammatorycells', 'SupplierPatientID', 'Yearofsamplecollection', 'PrimarySite', 'histology', 'PTNM_Radicality', 'Lymphaticinvasion', 'Venousinvasion', 'ERStatus', 'PRStatus', 'HER2Status', 'Pre-surgeryTx', 'Post-surgeryTx', 'Tag', 'Ptdiagnosis', 'DFSmonth', 'OSmonth', 'Comment', 'ER+DuctalCa', 'TripleNegDuctal', 'hormonesensitive', 'hormonerefractory', 'hormoneresistantaftersenstive', 'Fulvestran', 'microinvasion', 'I_plus_neg', 'SN', 'MIC', 'Count_Cells', 'FileName_FullStack', 'Height_FullStack', 'Width_FullStack', 'area', 'sum_area_cells', 'Subtype', 'HER2', 'ER

In [4]:
adata.obs['grade'].value_counts()

grade
3             174892
2             122318
1              98559
METASTASIS     31497
Name: count, dtype: int64

In [6]:
adata = adata[adata.obs['grade'] != 'METASTASIS'].copy()
adata

AnnData object with n_obs × n_vars = 395769 × 38
    obs: 'id', 'core', 'PID', 'location', 'grade', 'tumor_type', 'tumor_size', 'gender', 'menopausal', 'PTNM_M', 'age', 'Patientstatus', 'treatment', 'PTNM_T', 'DiseaseStage', 'PTNM_N', 'AllSamplesSVSp4.Array', 'TMALocation', 'TMAxlocation', 'yLocation', 'DonorBlocklabel', 'diseasestatus', 'TMABlocklabel', 'UBTMAlocation', '%tumorcells', '%normalepithelialcells', '%stroma', '%inflammatorycells', 'SupplierPatientID', 'Yearofsamplecollection', 'PrimarySite', 'histology', 'PTNM_Radicality', 'Lymphaticinvasion', 'Venousinvasion', 'ERStatus', 'PRStatus', 'HER2Status', 'Pre-surgeryTx', 'Post-surgeryTx', 'Tag', 'Ptdiagnosis', 'DFSmonth', 'OSmonth', 'Comment', 'ER+DuctalCa', 'TripleNegDuctal', 'hormonesensitive', 'hormonerefractory', 'hormoneresistantaftersenstive', 'Fulvestran', 'microinvasion', 'I_plus_neg', 'SN', 'MIC', 'Count_Cells', 'FileName_FullStack', 'Height_FullStack', 'Width_FullStack', 'area', 'sum_area_cells', 'Subtype', 'HER2', 'ER

In [7]:
adata.write('data/zurich_grade_no_met.h5ad')

In [1]:
x=1

In [2]:
import scanpy as sc

/lustre/groups/ml01/workspace/anastasia.litinetskaya/mambaforge/envs/sc/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/lustre/groups/ml01/workspace/anastasia.litinetskaya/mambaforge/envs/sc/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/lustre/groups/ml01/workspace/anastasia.litinetskaya/mambaforge/envs/sc/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/lustre/groups/ml01/workspace/anastasia.litinetskaya/mambaforge/envs/sc/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx ins

In [3]:
adata = sc.read('data/zurich_grade.h5ad')
adata

/lustre/groups/ml01/workspace/anastasia.litinetskaya/mambaforge/envs/sc/lib/python3.10/site-packages/anndata/__init__.py:42: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(


AnnData object with n_obs × n_vars = 427266 × 38
    obs: 'id', 'core', 'PID', 'location', 'grade', 'tumor_type', 'tumor_size', 'gender', 'menopausal', 'PTNM_M', 'age', 'Patientstatus', 'treatment', 'PTNM_T', 'DiseaseStage', 'PTNM_N', 'AllSamplesSVSp4.Array', 'TMALocation', 'TMAxlocation', 'yLocation', 'DonorBlocklabel', 'diseasestatus', 'TMABlocklabel', 'UBTMAlocation', '%tumorcells', '%normalepithelialcells', '%stroma', '%inflammatorycells', 'SupplierPatientID', 'Yearofsamplecollection', 'PrimarySite', 'histology', 'PTNM_Radicality', 'Lymphaticinvasion', 'Venousinvasion', 'ERStatus', 'PRStatus', 'HER2Status', 'Pre-surgeryTx', 'Post-surgeryTx', 'Tag', 'Ptdiagnosis', 'DFSmonth', 'OSmonth', 'Comment', 'ER+DuctalCa', 'TripleNegDuctal', 'hormonesensitive', 'hormonerefractory', 'hormoneresistantaftersenstive', 'Fulvestran', 'microinvasion', 'I_plus_neg', 'SN', 'MIC', 'Count_Cells', 'FileName_FullStack', 'Height_FullStack', 'Width_FullStack', 'area', 'sum_area_cells', 'Subtype', 'HER2', 'ER

In [6]:
adata.obs[['grade', 'core']].drop_duplicates()['grade'].value_counts()

grade
3             130
2             122
1              95
METASTASIS     12
Name: count, dtype: int64

In [7]:
adata.obs[['core']].drop_duplicates().value_counts()

core                      
ZTMA208_slide_1_Ay12x5        1
ZTMA208_slide_45_By11x5       1
ZTMA208_slide_50_Ay11x2       1
ZTMA208_slide_49_By3x1        1
ZTMA208_slide_49_By2x8        1
                             ..
ZTMA208_slide_23_Cy5x7_311    1
ZTMA208_slide_23_Cy5x7_310    1
ZTMA208_slide_23_Cy5x6        1
ZTMA208_slide_23_Cy5x5        1
ZTMA208_slide_72_By14x7       1
Name: count, Length: 359, dtype: int64

In [8]:
adata = sc.read('data/zurich_grade_no_met.h5ad')
adata

/lustre/groups/ml01/workspace/anastasia.litinetskaya/mambaforge/envs/sc/lib/python3.10/site-packages/anndata/__init__.py:42: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(


AnnData object with n_obs × n_vars = 395769 × 38
    obs: 'id', 'core', 'PID', 'location', 'grade', 'tumor_type', 'tumor_size', 'gender', 'menopausal', 'PTNM_M', 'age', 'Patientstatus', 'treatment', 'PTNM_T', 'DiseaseStage', 'PTNM_N', 'AllSamplesSVSp4.Array', 'TMALocation', 'TMAxlocation', 'yLocation', 'DonorBlocklabel', 'diseasestatus', 'TMABlocklabel', 'UBTMAlocation', '%tumorcells', '%normalepithelialcells', '%stroma', '%inflammatorycells', 'SupplierPatientID', 'Yearofsamplecollection', 'PrimarySite', 'histology', 'PTNM_Radicality', 'Lymphaticinvasion', 'Venousinvasion', 'ERStatus', 'PRStatus', 'HER2Status', 'Pre-surgeryTx', 'Post-surgeryTx', 'Tag', 'Ptdiagnosis', 'DFSmonth', 'OSmonth', 'Comment', 'ER+DuctalCa', 'TripleNegDuctal', 'hormonesensitive', 'hormonerefractory', 'hormoneresistantaftersenstive', 'Fulvestran', 'microinvasion', 'I_plus_neg', 'SN', 'MIC', 'Count_Cells', 'FileName_FullStack', 'Height_FullStack', 'Width_FullStack', 'area', 'sum_area_cells', 'Subtype', 'HER2', 'ER

In [13]:
adata.obs['PID'].nunique()

72

In [9]:
adata.obs[['grade', 'core']].drop_duplicates()['grade'].value_counts()

grade
3    130
2    122
1     95
Name: count, dtype: int64

In [10]:
adata.obs[['core']].drop_duplicates().value_counts()

core                      
ZTMA208_slide_1_Ay12x5        1
ZTMA208_slide_48_By3x4        1
ZTMA208_slide_50_Ay11x2       1
ZTMA208_slide_49_By3x1        1
ZTMA208_slide_49_By2x8        1
                             ..
ZTMA208_slide_23_Cy5x7_311    1
ZTMA208_slide_23_Cy5x7_310    1
ZTMA208_slide_23_Cy5x6        1
ZTMA208_slide_23_Cy5x5        1
ZTMA208_slide_72_By14x7       1
Name: count, Length: 347, dtype: int64